This notebook aim to save and combine data akademik dan data pedoman akademik

### Data Pedoman Akademik (Dense Method)

In [2]:
data_pedoman_dan_rule = {
    "data": [
        "https://akademik.nurulfikri.ac.id/1-satuan-kredit-semester-sks/",
        "https://akademik.nurulfikri.ac.id/1-syarat-kelulusan/",
        "https://akademik.nurulfikri.ac.id/2-aturan/",
        "https://akademik.nurulfikri.ac.id/3-kode-etik-mahasiswa/",
        "https://akademik.nurulfikri.ac.id/4-suasana-akademik/",
        "https://akademik.nurulfikri.ac.id/4-profil-dosen/",
        "https://akademik.nurulfikri.ac.id/1-sejarah/",
    ],
    "sumber_data": [
        "satuan kredit semester",
        "syarat kelulusan",
        "aturan dan kode etik",
        "kode etik mahasiswa",
        "suasana akademik",
        "profile dosen",
        "sejarah sttnf",
    ],
}

In [3]:
import os 


LANGSMITH_TRACING = os.getenv("LANGSMITH_TRACING")
LANGSMITH_API_KEY = os.getenv("LANGSMITH_API_KEY")
LANGSMITH_PROJECT = os.getenv("LANGSMITH_PROJECT")

In [4]:
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader(data_pedoman_dan_rule["data"])

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [5]:
pages = []

for doc in loader.lazy_load():
    pages.append(doc)

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
docs_splits = text_splitter.split_documents(pages)


### Model Bahasa (IndoBert)

In [7]:
# memanggil Indobert dari transformer
from transformers import BertTokenizer, AutoModel

tokenizer = BertTokenizer.from_pretrained("Indobenchmark/indobert-base-p1")
model = AutoModel.from_pretrained("indobenchmark/indobert-base-p1")

/Users/a/Programming/Langchain-Project/my-env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
# membuat class model embdding
from typing import List 
from langchain_core.embeddings import Embeddings
import torch

class IndoBertEmbeddings(Embeddings):
    def __init__(self, model_name="indobenchmark/indobert-base-p1"):
        self.tokenizer = BertTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name)
        self.model.eval()


    def _generate_embedding(self, text: str) -> List[float]:
        inputs = self.tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=512)

        with torch.no_grad():
            outputs = self.model(**inputs)

        # polling token menjadi satu vector kalimat
        token_embeddings = outputs.last_hidden_state

        # melakukan mean polling
        sentence_embeddings = token_embeddings.mean(dim=1)

        # konversi ke list python
        return sentence_embeddings.squeeze().tolist()
    

    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        return [self._generate_embedding(text) for text in texts]
    

    # metode untuk pencarian query pada chroma 
    def embed_query(self, text: str) -> List[float]:
        return self._generate_embedding(text)


In [14]:
embeddings = IndoBertEmbeddings()

In [10]:
from langchain_elasticsearch import ElasticsearchStore

In [17]:
vector_store = ElasticsearchStore(
    es_url="http://localhost:9200",
    index_name="langchain_index",
    embedding=embeddings,
    es_user="elastic",
    es_password="Xkhwf3uB",
)

In [18]:
vector_store.add_documents(docs_splits)

['2f528aed-fb07-4c89-a6d6-70a24c593489',
 'a2067f40-925f-41c2-8c91-070397d2a7d7',
 '0f74cdf4-6edd-4b1b-b8e9-f2ac54bda2e5',
 'b2e7bfae-b08c-4e83-93f7-df8b939b8030',
 '4b2b566c-50db-4c0a-a2db-0167a2900be2',
 '75138e21-bbdf-435a-9b17-59f8e9166133',
 '87af86cd-bb50-4b2d-8952-c076a5ff8ba9',
 '7887d956-e8f6-4d88-b8a8-4922ed9f675e',
 'c8445a93-00c3-4b5c-a379-742ef2c6e5e5',
 'cd0fe068-c19b-45e4-ad2f-bf6840a58b28',
 '4a0bfbe8-c70a-4c1d-ab61-a3d5bb2071c7',
 '0f0e41e7-627c-47d9-bd79-984bee1c7f50',
 'dd0d757e-da54-4c98-9c89-a541826b4e5d',
 '41ca1167-cbcb-47f4-90c8-4ea66a9b1b2c',
 '06f612ad-7b93-47b2-8fae-c07dde8a2d0d',
 'be7ce6cf-a536-4639-b936-646370ddb915',
 'd3043e1d-4994-468f-a4d4-7db0356cfcff',
 '366f7a4e-70a3-4c8f-b48f-66face56902b',
 'd518d6b2-3540-458b-8812-41714f542701',
 '30ed562e-ccdd-4e1c-baa3-69177bb0bfec',
 '339e431b-4b5e-46ce-9b9c-e5a63df821c6',
 '827705ba-5509-4de9-8ff3-d046d150d603',
 'ff7169f4-6580-4d2d-a7ae-81c89096e10c',
 'f8b3e3a0-cbfa-4a67-88c9-0e41b61233f7',
 'b92d938c-8446-

### Testing Dense Retriever

In [20]:
retriever = vector_store.as_retriever(
    search_type="similarity_score_threshold", search_kwargs={"score_threshold": 0.2}
)


In [21]:
retriever.invoke("Berapa batas minimum IPK agar mahasiswa dinyatakan lulus?")

[Document(metadata={'source': 'https://akademik.nurulfikri.ac.id/1-satuan-kredit-semester-sks/', 'title': '1. Satuan Kredit Semester (SKS) – Pedoman Akademik STT-NF', 'language': 'en-US'}, page_content='Masa Studi\nKetentuan masa studi adalah sebagai berikut:\n\nMasa studi adalah masa untuk penyelesaian beban studi dalam mengikuti proses pendidikan pada program studinya.\nProgram sarjana harus diselesaikan dalam waktu tidak lebih dari tujuh tahun (14 Semester), terhitung mulai saat mahasiswa terdaftar sebagai mahasiswa. Jika ternyata sampai batas masa studi yang ditentukan, mahasiswa belum dapat menyelesaikan studi sarjananya, maka yang bersangkutan dinyatakan tidak mampu melanjutkan studinya/ Drop Out (DO).\nMasa studi tujuh tahun tersebut termasuk cuti akademik dan bagi mahasiswa yang tidak mendaftar ulang per semester tetap diperhitungkan sebagai masa studi.\nBagi mahasiswa yang melampaui masa studi empat tahun (8 Semester) akan diberlakukan ketentuan SPP Progresif.\n\n\n\n\nLast Mo

### Data Akademik Mahasiswa (Sparse Method)

In [22]:
data_akademik = [
    '/Users/a/Programming/Langchain-Project/external-data/sintetik-data-akademik-mahasiswa (2).xlsx'
]

### Processing document

In [31]:
from typing import Iterator
from langchain_core.document_loaders import BaseLoader
from langchain_core.documents import Document as LCDocument
from docling.document_converter import DocumentConverter

In [32]:
class DoclingLoader(BaseLoader):
    def __init__(self, file_path: str | list[str]) -> None:
        self._file_paths = file_path if isinstance(file_path, list) else [file_path]
        self._converter = DocumentConverter()

    def lazy_load(self) -> Iterator[LCDocument]:
        for source in self._file_paths:
            dl_doc = self._converter.convert(source).document
            text = dl_doc.export_to_markdown()
            yield LCDocument(page_content=text)

In [35]:
loader = DoclingLoader(data_akademik)

docs_akademik = loader.lazy_load()

In [37]:
data_akademik_split = text_splitter.split_documents(docs_akademik)

2026-02-03 16:53:29,487 - INFO - detected formats: [<InputFormat.XLSX: 'xlsx'>]
2026-02-03 16:53:29,542 - INFO - Going to convert document batch...
2026-02-03 16:53:29,542 - INFO - Initializing pipeline for SimplePipeline with options hash 995a146ad601044538e6a923bea22f4e
2026-02-03 16:53:29,774 - WARNING - The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.
2026-02-03 16:53:29,775 - INFO - Loading plugin 'docling_defaults'
2026-02-03 16:53:29,778 - INFO - Registered picture descriptions: ['vlm', 'api']
2026-02-03 16:53:29,778 - INFO - Processing document sintetik-data-akademik-mahasiswa (2).xlsx
2026-02-03 16:53:29,779 - INFO - Processing sheet 0: Data Mahasiswa Sintetik untuk R
2026-02-03 16:53:29,785 - INFO - Finished converting document sintetik-data-akademik-mahasiswa (2).xlsx in 0.30 sec.


In [38]:
vector_store_sparse = ElasticsearchStore(
    es_url="http://localhost:9200",
    index_name="test_index",
    es_user="elastic",
    es_password="Xkhwf3uB",
    strategy=ElasticsearchStore.BM25RetrievalStrategy(),
)

2026-02-03 16:53:44,149 - INFO - GET http://localhost:9200/ [status:200 duration:0.007s]


In [39]:
vector_store_sparse.add_documents(data_akademik_split)

2026-02-03 16:54:14,088 - INFO - HEAD http://localhost:9200/test_index [status:404 duration:0.008s]
2026-02-03 16:54:14,250 - INFO - PUT http://localhost:9200/test_index [status:200 duration:0.159s]
2026-02-03 16:54:14,291 - INFO - PUT http://localhost:9200/_bulk?refresh=true [status:200 duration:0.039s]


['2637d8e0-3c58-4dd7-b771-86dd6588c819',
 '9d5620a2-d8b3-4a2a-85ac-3efb041d2f4e',
 '908a182d-6c4a-4839-8978-b3fe3ac226dc',
 'f07cb45d-ae29-42eb-9f20-0144aecc002f',
 '8dcb4d49-1014-451b-b728-e532e48d5e55',
 'b40b1f28-8a8c-4e52-acc3-92017d30acef',
 'ae6c2241-26ad-4c09-898e-7a3623d24bc1',
 'dd5a0e25-9c9e-49de-ba33-216c31894fc8',
 'f8c46d3d-1a56-4afc-8b7c-5253aeab6582',
 '06c57b98-51a5-41e5-bf5e-f6afb349ff61',
 '181572b3-6a32-4ae5-b31d-9d45bcc80493',
 'd0c4cc1f-4f0f-4a4c-a422-6aaa345b4fe6',
 '563819a7-fe19-4abe-a0de-ca87f101c424',
 '89c0abd7-2ff3-489b-9215-e5fa557981e3',
 '0f585606-27f1-4028-8000-b307f2edd470',
 '78232480-6340-47d3-b7bf-cc6d8112bc53',
 'fba0e816-1917-42b5-a31b-4496012f7dde']

In [41]:
vector_store_sparse.similarity_search("berapa ipk romi wahyudi")

2026-02-03 16:54:58,040 - INFO - POST http://localhost:9200/test_index/_search?_source_includes=metadata,text [status:200 duration:0.031s]


[Document(metadata={}, page_content='|   17 | 2.021e+07   | Qori Handayani        | T. Informatika   |          5 |          20 |  3.15 |  3.2  |              87 | Aktif     |\n|   18 | 2.021e+07   | Romi Wahyudi Hasibuan | T. Informatika   |          5 |          18 |  2.5  |  2.8  |              75 | Aktif     |\n|   19 | 2.021e+07   | Siti Aminah           | T. Informatika   |          5 |          24 |  4    |  3.95 |              99 | Aktif     |\n|   20 | 2.021e+07   | Tono Suherman         | T. Informatika   |          5 |          15 |  0.5  |  1.8  |              30 | Non-Aktif |\n|   21 | 2.021e+07   | Usman Affandi         | T. Informatika   |          5 |          20 |  3.05 |  3.1  |              85 | Aktif     |\n|   22 | 2.021e+07   | Vina Panduwinata      | T. Informatika   |          5 |          22 |  3.55 |  3.6  |              93 | Aktif     |\n|   23 | 2.021e+07   | Wahyu Hidayat         | T. Informatika   |          5 |          20 |  2.85 |  3    |              8